In [2]:
"""Program to simulate the X-ray spectra and color of NS and CV spectra.

Functions needed:
1. Choose a random source and collect the corresponding response and background
2. A random distribution of NH and power-law for the NS spectra.
3. Random distribution of NH, power-law and Fe-line eq width for CV
4. Fake-it simulations.
Analysis should be done both for PN and MOS
Use functions with '_fromsrc' to generate spectra with background.
"""


import copy
import glob2
import os
import xspec
import numpy as np
from astropy.io import fits

In [3]:
def get_resp(src_file, obs_folder='./', rmf_folder='./'):
    """Get background and responses for the given source file.

    Inputs:
    src_file = Combined source file
    obs_folder - Folder containing individual observations of the source
    """
    spectra_header = fits.open(src_file)[1].header
    bg_file = obs_folder + spectra_header['BACKFILE']
    arf_file = obs_folder + spectra_header['ANCRFILE']
    resp_file = rmf_folder + spectra_header['RESPFILE']
    exptime = spectra_header['EXPOSURE']
    file_list = [bg_file, arf_file, resp_file]
    for i, file in enumerate(file_list):
        if file.split('/')[-1] == 'None':
            file_list[i] = ''
    return file_list[0], file_list[1], file_list[2], exptime

In [4]:
def sim_msp(resp_file, arf_file, bg_file, exp_s, sim_msp_name, nh_val,
            gamma_val, unabs_lx_val):
    """Simulate the MSP spectra with the given values.

    CANNOT account the background properly
    """
    msp_settings = xspec.FakeitSettings(
        response=resp_file, arf=arf_file, background=bg_file, exposure=exp_s,
        fileName=sim_msp_name)
    msp_model = xspec.Model('tbabs*pegpwrlw')
    unabs_flux = unabs_lx_val/(7.65757E+45)
    msp_model.setPars({1: nh_val/1.0E+22, 2: gamma_val, 3: 2, 4: 10,
                       5: unabs_flux/1.0E-12})
    xspec.AllData.fakeit(1, msp_settings)
    xspec.AllData.clear()
    xspec.AllModels.clear()

In [5]:
def sim_msp_from_src(src_file, sim_msp_name, nh_val, gamma_val,
                     unabs_lx_val):
    """Simulate MSP spectra from a source spectra.

    Cannot account for the background.
    """
    msp_settings = xspec.FakeitSettings(fileName=sim_msp_name)
    spectrum = xspec.Spectrum(src_file)
    msp_model = xspec.Model('tbabs*pegpwrlw')
    unabs_flux = unabs_lx_val/(7.65757E+45)
    msp_model.setPars({1: nh_val/1.0E+22, 2: gamma_val, 3: 2, 4: 10,
                       5: unabs_flux/1.0E-12})
    xspec.AllData.fakeit(1, msp_settings)
    xspec.AllData.clear()
    xspec.AllModels.clear()

In [6]:
def sim_cv_from_src(src_file, sim_cv_name, nh_val, temp_val, unabs_lx_val,
                    ew_64, ew_67, ew_70):
    """Simulate spectra of CVs.
    
    Use this for simulations that need background. The background from the
    src_file will be used for the fake spectra
    """
    # Input file name of fake spectra
    ip_settings = xspec.FakeitSettings(fileName=sim_cv_name)
    spectrum = xspec.Spectrum(src_file)   # Load source file of the background
    # Switch off thermal lines in the thermal plasma. Will be added later.
    xspec.Xset.addModelString("APECNOLINES", "yes") 
    ip_model = xspec.Model("tbabs*(apec+gaussian+gaussian+gaussian)")
    unabs_flux = unabs_lx_val/(7.65757E+45)    # Flux at GC distance
    ip_model.setPars({1: nh_val, 2: temp_val, 6: 6.4, 8: 1.0E-4, 9: 6.7,
                      11: 1.0E-4, 12: 7.0, 14: 1.0E-4})  # setting parameters
    # No way to directly specify equivalent width in the model, therefore
    # calculate the equivalenth width given the model and then scale the norm
    # of the model
    xspec.AllModels.eqwidth(3, rangeFrac=0.0)
    test_ew_64 = spectrum.eqwidth[0]
    xspec.AllModels.eqwidth(4, rangeFrac=0.0)
    test_ew_67 = spectrum.eqwidth[0]
    xspec.AllModels.eqwidth(5, rangeFrac=0.0)
    test_ew_70 = spectrum.eqwidth[0]
    norm_64 = ew_64/test_ew_64*1.0E-4
    print(norm_64)
    norm_67 = ew_67/test_ew_67*1.0E-4
    print(norm_67)
    norm_70 = ew_70/test_ew_70*1.0E-4
    xspec.AllModels.clear()
    # Reset the parameters again with the calculated norms
    print(norm_70)
    ip_model = xspec.Model("tbabs*cflux*(apec+gaussian+gaussian+gaussian)")
    ip_model.setPars({1: nh_val, 2: 2.0, 3: 10.0, 4: np.log10(unabs_flux),
                      5: temp_val, 9: 6.4, 11: norm_64, 12: 6.7, 14: norm_67,
                      15: 7.0, 17: norm_70})
    xspec.AllData.fakeit(1, ip_settings)
    xspec.AllData.clear()
    xspec.AllModels.clear()

In [7]:
def sim_cv_from_craig(src_file, sim_msp_name, nh_val, gamma_val, unabs_lx_val,
                      ew_fe):
    """This was from one of Craig's results, where is used one Gaussian."""
    ip_settings = xspec.FakeitSettings(fileName=sim_msp_name)
    spectrum = xspec.Spectrum(src_file)
    '''
    ip_model = xspec.Model("tbabs*(pegpwrlw+gaussian)")
    unabs_flux = unabs_lx_val/(7.65757E+45)
    ip_model.setPars({1: nh_val, 2: gamma_val, 5: unabs_flux/1.0E-12, 6: 6.54,
                      7: 0.2, 8: 1.0E-4})
    xspec.AllModels.eqwidth(3, rangeFrac=0.0)
    test_ew = spectrum.eqwidth[0]
    norm = ew_fe/test_ew*1.0E-4
    xspec.AllModels.clear()
    '''
    ip_model = xspec.Model("tbabs*(pegpwrlw+gaussian+)")
    ip_model.setPars({1: nh_val, 2: gamma_val, 5: unabs_flux/1.0E-12, 6: 6.54,
                      7: 0.2, 8: norm})
    xspec.AllData.fakeit(1, ip_settings)
    xspec.AllData.clear()
    xspec.AllModels.clear()

In [8]:
def msp_simulations(num_msps, nh_vals, gamma_vals, unabs_lx_vals,
                    resp_files=None, arf_files=None, bg_files=None,
                    obs_folder=None, rmf_folder=None, exp_times=None,
                    sim_msp_folder='./'):
    """Simulate MSPs."""
    if resp_files is None:
        if obs_folder is None or rmf_folder is None:
            raise ValueError('Either give the response files explicitly or' +
                             'give the observation and rmf folders from which'
                             + 'they should be sampled')
        src_files = glob2.glob(obs_folder + '*_src_grp.ds')
        src_files_forsim = np.random.choice(src_files, size=num_msps)

    for i, msp_num in enumerate(num_msps):
        if resp_files is None:
            bg_file, arf_file, resp_file, exptime = get_resp(
                src_files_forsim[i], obs_folder=obs_folder,
                rmf_folder=rmf_folder)
        else:
            bg_file = bg_files[i]
            arf_file = arf_files[i]
            resp_file = resp_files[i]
            exptime = exp_times[i]

        sim_msp(resp_file, arf_file, bg_file, exptime,
                sim_msp_folder+'msp_'+str(i)+'.fak', nh_vals[i], gamma_vals[i],
                unabs_lx_vals[i])
        if i % 1000 == 0:
            print('Finished ' + str(i) + 'simulations')


In [9]:
def get_xmm_src_files(src_folder):
    """Get PN and MOS src files."""
    pn_src_files = glob2.glob(src_folder + '*_PN_*src_grp1*cts.ds')
    mos_src_files = glob2.glob(src_folder + '*_MOS_*src_grp1*cts.ds')
    # pn_mos_png_files = glob2.glob(src_folder + '*_PN_MOS_combined_src.png')
    common_files_pn = []
    common_files_mos = []
    only_pn_files = copy.copy(pn_src_files)
    only_mos_files = copy.copy(mos_src_files)
    for pn_file in pn_src_files:
        pn_src_num = pn_file.split('/')[-1].split('_')[0]
        if os.path.isfile(src_folder + pn_src_num +
                          '_MOS_combined_src_grp.ds'):
            mos_file = glob2.glob(src_folder + pn_src_num +
                                  '_MOS_combined_src_grp1*cts.ds')[0]
            common_files_pn.append(pn_file)
            common_files_mos.append(mos_file)
            only_pn_files.remove(pn_file)
            only_mos_files.remove(mos_file)
    return common_files_pn, common_files_mos, only_pn_files, only_mos_files

In [10]:
def get_src_nums(src_file_list):
    src_nums = []
    for src_file in src_file_list:
        src_num = src_file.split('/')[-1].split('_')[0]
        src_nums.append(src_num)
    return src_nums

In [11]:
def msp_sims_chandra_src(num_msps, nh_vals, gamma_vals, unabs_lx_vals,
                         src_folder, sim_msp_folder='./', file_prefix='msp_'):
    """Simulate Chandra MSPs.
    
    Give the absolute path for sim_msp_folder.
    """
    src_files = glob2.glob(src_folder + '/*/*_combined_src.pi')
    src_args = np.random.choice(np.arange(len(src_files)), size=num_msps)
    for i in range(num_msps):
        sim_msp_from_src(
            src_files[src_args[i]],
            sim_msp_folder + file_prefix + str(i) + '.fak',
            nh_vals[i], gamma_vals[i], unabs_lx_vals[i])
        
        if i % 1000 == 0:
            print('Finished ' + str(i) + ' simulations')


In [12]:
def cvs_sims_chandra_src(num_msps, nh_vals, temp_vals, unabs_lx_vals,
                         ew_64_vals, ew_67_vals, ew_70_vals, src_folder,
                         sim_cv_folder='./', file_prefix='ip_'):
    """Simulate Chandra CVs.
    
    Give the absolute path for sim_msp_folder.
    """
    src_files = glob2.glob(src_folder + '/*/*_combined_src.rmf')
    src_args = np.random.choice(np.arange(len(src_files)), size=num_msps)
    curr_dir = os.getcwd()
    for i in range(num_msps):
        os.chdir(os.path.dirname(src_files[src_args[i]])) 
        srcfilename = os.path.basename(src_files[src_args[i]])[:-3] + 'pi'
        sim_cv_from_src(
            srcfilename,
            sim_cv_folder + file_prefix + str(i) + '.fak',
            nh_vals[i], temp_vals[i], unabs_lx_vals[i], ew_64_vals[i],
            ew_67_vals[i], ew_70_vals[i])
        os.chdir(curr_dir)
        
        if i % 1000 == 0:
            print('Finished ' + str(i) + ' simulations')


In [13]:
def msp_sims_from_src2(num_msps, nh_vals, gamma_vals, unabs_lx_vals,
                       src_folder, sim_msp_folder='./', file_prefix='msp_'):
    """Simulate equal number of PN and MOS MSPs from source files."""
    (common_files_pn, common_files_mos, only_pn_files,
     only_mos_files) = get_xmm_src_files(src_folder)
    pn_files = common_files_pn + only_pn_files
    mos_files = common_files_mos + only_mos_files
    pn_src_args = np.random.choice(np.arange(len(pn_files)), size=num_msps)
    mos_src_args = np.random.choice(np.arange(len(mos_files)), size=num_msps)
    for i in range(num_msps):
        sim_msp_from_src(
                pn_files[pn_src_args[i]],
                sim_msp_folder + file_prefix + str(i) + '_PN.fak',
                nh_vals[i], gamma_vals[i], unabs_lx_vals[i])
        sim_msp_from_src(
                mos_files[mos_src_args[i]],
                sim_msp_folder + file_prefix + str(i) + '_MOS.fak',
                nh_vals[i], gamma_vals[i], unabs_lx_vals[i])

        if i % 1000 == 0:
            print('Finished ' + str(i) + ' simulations')
    

In [14]:
def msp_sims_from_src(num_msps, nh_vals, gamma_vals, unabs_lx_vals, src_folder,
                      sim_msp_folder='./', file_prefix='msp_'):
    """Simulate PN and MOS MSPs from source files based on source detection."""
    (common_files_pn, common_files_mos, only_pn_files,
     only_mos_files) = get_xmm_src_files(src_folder)
    src_files = common_files_pn + only_pn_files + only_mos_files
    src_args_forsim = np.random.choice(
        np.arange(len(src_files)), size=num_msps)
    for i in range(num_msps):
        if src_args_forsim[i] < len(common_files_pn) + len(only_pn_files):
            sim_msp_from_src(
                src_files[src_args_forsim[i]],
                sim_msp_folder + file_prefix + str(i) + '_PN.fak',
                nh_vals[i], gamma_vals[i], unabs_lx_vals[i])
            if src_args_forsim[i] < len(common_files_pn):
                sim_msp_from_src(
                    common_files_mos[src_args_forsim[i]],
                    sim_msp_folder + file_prefix + str(i) + '_MOS.fak',
                    nh_vals[i], gamma_vals[i], unabs_lx_vals[i])
        else:
            sim_msp_from_src(
                src_files[src_args_forsim[i]],
                sim_msp_folder + file_prefix + str(i) + '_MOS.fak',
                nh_vals[i], gamma_vals[i], unabs_lx_vals[i])

        if i % 1000 == 0:
            print('Finished ' + str(i) + ' simulations')

In [15]:
def cvs_sims_from_src(num_cvs, nh_vals, temp_vals, unabs_lx_vals, ew_64_vals,
                      ew_67_vals, ew_70_vals, src_folder, sim_cv_folder='./',
                      file_prefix='cv_'):
    """"Simulate CVs from source files."""
    (common_files_pn, common_files_mos, only_pn_files,
     only_mos_files) = get_xmm_src_files(src_folder)
    src_files = common_files_pn + only_pn_files + only_mos_files
    src_args_forsim = np.random.choice(
        np.arange(len(src_files)), size=num_cvs)
    for i in range(num_cvs):
        if src_args_forsim[i] < len(common_files_pn) + len(only_pn_files):
            sim_cv_from_src(
                src_files[src_args_forsim[i]],
                sim_cv_folder + file_prefix + str(i) + '_PN.fak', nh_vals[i],
                temp_vals[i], unabs_lx_vals[i], ew_64_vals[i], ew_67_vals[i],
                ew_70_vals[i])
            if src_args_forsim[i] < len(common_files_pn):
                sim_cv_from_src(
                    common_files_mos[src_args_forsim[i]],
                    sim_cv_folder + file_prefix + str(i) + '_MOS.fak',
                    nh_vals[i], temp_vals[i], unabs_lx_vals[i], ew_64_vals[i],
                    ew_67_vals[i], ew_70_vals[i])
        else:
            sim_cv_from_src(
                src_files[src_args_forsim[i]],
                sim_cv_folder + file_prefix + str(i) + '_MOS.fak', nh_vals[i],
                temp_vals[i], unabs_lx_vals[i], ew_64_vals[i], ew_67_vals[i],
                ew_70_vals[i])

        if i % 1000 == 0:
            print('Finished ' + str(i) + ' simulations')

In [16]:
def cvs_sims_from_src2(num_msps, nh_vals, temp_vals, unabs_lx_vals, ew_64_vals,
                       ew_67_vals, ew_70_vals, src_folder, sim_cv_folder='./',
                       file_prefix='cv_'):
    """Simulate equal number of PN and MOS CVs from source files."""
    (common_files_pn, common_files_mos, only_pn_files,
     only_mos_files) = get_xmm_src_files(src_folder)
    pn_files = common_files_pn + only_pn_files
    mos_files = common_files_mos + only_mos_files
    pn_src_args = np.random.choice(np.arange(len(pn_files)), size=num_msps)
    mos_src_args = np.random.choice(np.arange(len(mos_files)), size=num_msps)
    for i in range(num_msps):
        sim_cv_from_src(
                pn_files[pn_src_args[i]],
                sim_cv_folder + file_prefix + str(i) + '_PN.fak',
                nh_vals[i], temp_vals[i], unabs_lx_vals[i], ew_64_vals,
                ew_67_vals[i], ew_70_vals[i])
        sim_cv_from_src(
                mos_files[mos_src_args[i]],
                sim_cv_folder + file_prefix + str(i) + '_MOS.fak',
                nh_vals[i], temp_vals[i], unabs_lx_vals[i], ew_64_vals[i],
                ew_67_vals[i], ew_70_vals[i])

        if i % 1000 == 0:
            print('Finished ' + str(i) + ' simulations')

In [17]:
def get_msp_param_vals(num_msps, nh_abs_type):
    """Get parameter values for the MSP simulations."""
    if nh_abs_type == 'high':
        nh_vals = np.random.uniform(22.7, 23.7, num_msps)
    elif nh_abs_type == 'mid':
        nh_vals = np.random.uniform(22.0, 22.7, num_msps)
    elif nh_abs_type == 'low':
        nh_vals = np.random.uniform(21.0, 22.0, num_msps)
    else:
        print("'nh_abs_type' should be 'high', 'mid', or 'low'.")
    nh_vals = 10**nh_vals
    gamma_vals = np.random.uniform(1.0, 2.0, num_msps)
    lx_vals = np.random.uniform(31.0, 34.0, num_msps)
    lx_vals = 10**lx_vals
    return nh_vals, gamma_vals, lx_vals

In [18]:
def cv_param_vals(num_cvs, nh_abs_type, cv_type='IP'):
    """"Get parameter values for IP simulations.
    
    nh_vals = Currently using a log uniform distribution
    lx_vals - Currently using a log uniform distribution
    temp_vals, ew_vals - Using normal distribution. Might need to change

    """
    if nh_abs_type == 'high':
        nh_vals = np.random.uniform(22.7, 23.7, num_cvs)
    elif nh_abs_type == 'mid':
        nh_vals = np.random.uniform(22.0, 22.7, num_cvs)
    elif nh_abs_type == 'low':
        nh_vals = np.random.uniform(21.0, 22.0, num_cvs)
    else:
        print("'nh_abs_type' should be 'high', 'mid', or 'low'.")
    nh_vals = 10**nh_vals
    lx_vals = np.random.uniform(31.0, 34.0, num_cvs)
    lx_vals = 10**lx_vals
    if cv_type == 'IP':
        temp_vals = np.random.normal(34.0, 14.61, num_cvs)
        ew_64_vals = np.random.normal(115.0, 36.22, num_cvs)*2
        ew_67_vals = np.random.normal(107, 65.39, num_cvs)*2
        ew_70_vals = np.random.normal(80, 27.91, num_cvs)*2
    elif cv_type == 'SS':
        temp_vals = np.random.normal(27.2, 20.8, num_cvs)
        ew_64_vals = np.random.normal(280, 90, num_cvs)
        ew_67_vals = np.random.normal(241, 78.3, num_cvs)
        ew_70_vals = np.random.normal(91, 20.1, num_cvs)
    else:
        print('CV types can only be IP or SS')
    return nh_vals, temp_vals, lx_vals, ew_64_vals, ew_67_vals, ew_70_vals

In [19]:
def ip_param_vals(num_cvs, nh_abs_type='high'):
    """Get IP parameter values.

    Same as previous, but picking the temp and EW values randomly from the list
    rather than using a normal distribution.
    """
    if nh_abs_type == 'high':
        nh_vals = np.random.uniform(22.7, 23.7, num_cvs)
    elif nh_abs_type == 'mid':
        nh_vals = np.random.uniform(22.0, 22.7, num_cvs)
    elif nh_abs_type == 'low':
        nh_vals = np.random.uniform(21.0, 22.0, num_cvs)
    else:
        print("'nh_abs_type' should be 'high', 'mid', or 'low'.")
    nh_vals = 10**nh_vals
    lx_vals = np.random.uniform(31.0, 34.0, num_cvs)
    lx_vals = 10**lx_vals

    temp_vals = np.random.choice(
        [19.7, 42.6, 9.41, 19.1, 30.5, 63.6, 15.8, 43.5, 32.6, 65.9, 40.5,
         26.9, 26.6, 22.8, 47.3, 39.6, 31.6], num_cvs)
    ew_64_vals = np.random.choice(
        [158, 102, 32, 133, 128, 88, 139, 156, 128, 172, 120, 88, 97, 140, 131,
         70], num_cvs)/1000
    ew_67_vals = np.random.choice(
        [174, 81, 325, 73, 91, 59, 101, 116, 68, 79, 131, 121, 71, 102, 60,
         94], num_cvs)/1000
    ew_70_vals = np.random.choice(
        [100, 54, 110, 58, 32, 62, 134, 120, 62, 91, 104, 94, 70, 93, 69, 57],
        num_cvs)/1000
    return nh_vals, temp_vals, lx_vals, ew_64_vals, ew_67_vals, ew_70_vals

In [20]:
def ip_param_vals_craig(num_cvs):
    """Param values of ASCA IPs."""
    nh_vals = 10**np.random.uniform(22.7, 23.7, num_cvs)
    lx_vals = 10**np.random.uniform(31.0, 34.0, num_cvs)
    gamma_vals = np.random.choice(
        [2.53, 1.32, 0.9, 1.32, 1.98, 0.66, 1.29, 0.59, 1.08, 1.0, 1.11, 1.83,
         0.81, 1.07, 0.86, 1.2, 0.95, 1.49, 1.12, 1.96], num_cvs)
    ew_vals = np.random.choice(
        [769, 450, 4.12, 264, 772, 488, 206, 596, 311, 388, 403, 456, 298, 282,
         247, 389, 411, 691, 206, 698], num_cvs)/1000
    return nh_vals, gamma_vals, lx_vals, ew_vals


In [21]:
def main(num_msps=10000, nh_abs_type='high', src_folder=None,
         sim_msp_folder='./'):
    """Main function"""
    nh_vals, gamma_vals, lx_vals = get_msp_param_vals(num_msps, nh_abs_type)
    if src_folder is None:
        src_folder = './Galactic_' + nh_abs_type + 'NH_combinedXMM/'
    msp_sims_from_src(10000, nh_vals, gamma_vals, lx_vals, src_folder,
                      sim_msp_folder, 'msp_'+nh_abs_type+'NH_')
    msp_param_vals = np.column_stack(nh_vals, gamma_vals, lx_vals)
    np.savetxt(sim_msp_folder + 'paramfile.txt', msp_param_vals)
    return nh_vals, gamma_vals, lx_vals

In [22]:
def sim_cv_from_src(src_file, sim_cv_name, nh_val, temp_val, unabs_lx_val,
                    ew_64, ew_67, ew_70):
    """Simulate spectra of CVs.
    
    Use this for simulations that need background. The background from the
    src_file will be used for the fake spectra
    """
    # Input file name of fake spectra
    ip_settings = xspec.FakeitSettings(fileName=sim_cv_name)
    spectrum = xspec.Spectrum(src_file)   # Load source file of the background
    # Switch off thermal lines in the thermal plasma. Will be added later.
    xspec.Xset.addModelString("APECNOLINES", "yes") 
    ip_model = xspec.Model("tbabs*(apec+gaussian+gaussian+gaussian)")
    unabs_flux = unabs_lx_val/(7.65757E+45)    # Flux at GC distance
    ip_model.setPars({1: nh_val, 2: temp_val, 6: 6.4, 8: 1.0E-4, 9: 6.7,
                      11: 1.0E-4, 12: 7.0, 14: 1.0E-4})  # setting parameters
   
    
    # No way to directly specify equivalent width in the model, therefore
    # calculate the equivalenth width given the model and then scale the norm
    # of the model
    xspec.AllModels.eqwidth(3, rangeFrac=0.0)
    test_ew_64 = spectrum.eqwidth[0]
    xspec.AllModels.eqwidth(4, rangeFrac=0.0)
    test_ew_67 = spectrum.eqwidth[0]
    xspec.AllModels.eqwidth(5, rangeFrac=0.0)
    test_ew_70 = spectrum.eqwidth[0]
    norm_64 = ew_64/test_ew_64*1.0E-4
    print(norm_64)
    norm_67 = ew_67/test_ew_67*1.0E-4
    print(norm_67)
    norm_70 = ew_70/test_ew_70*1.0E-4
    xspec.AllModels.clear()
    # Reset the parameters again with the calculated norms
    print(norm_70)
    ip_model = xspec.Model("tbabs*cflux*(apec+gaussian+gaussian+gaussian)")
    ip_model.setPars({1: nh_val, 2: 2.0, 3: 10.0, 4: np.log10(unabs_flux),
                      5: temp_val, 9: 6.4, 11: norm_64, 12: 6.7, 14: norm_67,
                      15: 7.0, 17: norm_70})
   
    
    xspec.AllData.fakeit(1, ip_settings)
    xspec.AllData.clear()
    xspec.AllModels.clear()

In [224]:
def sim_cv_from_jasmine(src_file, sim_cv_name, nh_val, gamma_val, flux_2_10,
                        norm1, norm2, norm3, output_dir):
    
    """This generates the spectra of one CV with three Gaussian Fe lines."""
    full_path = os.path.join(output_dir, sim_cv_name)
    #cv_settings = xspec.FakeitSettings(fileName=sim_cv_name)
    cv_settings = xspec.FakeitSettings(fileName=full_path)
    cv_spectrum = xspec.Spectrum(src_file)  #Loads source data
    cv_model = xspec.Model("tbabs*(pegpwrlw+gaussian+gaussian+gaussian)")  #Creates model
    cv_model.setPars({1: nh_val, 
                      2: gamma_val, 
                      3: 2.0, 
                      4: 10.0, 
                      5: flux_2_10,  #pegpwrlw norm, flux from 2 keV to 10 keV
                      6: 6.40, 7: 0.0, 8: norm1,  #Fe line at 6.4 keV
                      9: 6.70, 10: 0.0, 11: norm2,  #Fe line at 6.7 keV
                      12: 6.90, 13: 0.0, 14: norm3})  #Fe line at 6.9 keV
    
    norm_64 = norm1
    norm_67 = norm2
    norm_69 = norm3
    '''
    #Generate normalizations for the 6.4, 6.7 and 6.9 keV Fe emission lines
    norm_64 = ratio_64_67 * I_67 * 1E-6 #Ratio of I_64 and I_67 comes from Xu 2016
    norm_67 = I_67 * 1E-6  #I_67 and I_69 comes from Mondal 2025
    norm_69 = I_69 * 1E-6
    '''
    #Print check
    print("Computed gaussian norms (phot/cm^2/s):")
    print(" norm_6.4 =", norm_64)
    print(" norm_6.7 =", norm_67)
    print(" norm_6.9 =", norm_69)
    
    
    xspec.AllModels.clear()  #Clear model 
    '''
    
    cv_model = xspec.Model("tbabs*(pegpwrlw+gaussian+gaussian+gaussian)")  
    cv_model.setPars({1: nh_val, 
                      2: gamma_val,
                      3: 2.0, 
                      4: 10.0, 
                      5: flux_2_10/1E-12,
                      6: 6.4,  8: norm_64,  #Gaussian at 6.4 keV
                      9: 6.7, 11: norm_67,  #Gaussian at 6.7 keV
                      12: 6.9, 14: norm_69})  #Gaussian at 6.9 keV
    '''
    xspec.AllData.fakeit(1, cv_settings)
    xspec.AllData.clear()
    xspec.AllModels.clear()
    

In [225]:
def cv_paramvals_mondal(num_cvs):
    """
    This function returns arrays of nh, lx, gamma, and Fe line intensity values for
    a given number (ie. num_cvs) of CVs.
    """
    #add empty lists to append values to call them in next function
    nh_table_vals = []
    flux_table_vals = []
    gamma_table_vals = []
    I_67_table_vals = []
    I_69_table_vals = []
    I_64_table_vals = []

    """
    These arrays contain CV data from Mondal 2025 and Xu 2016. multiply intensities by 10^-6, flux 10^-13, 
    """
    nh_table      = np.array([1.07, 0.67, 3.72, 0.7, 0.45, 0.81, 0.12, 4.53, 2.58, 0.42, 6.22, 3.36, 1.46, 0.24, 0.76, 0.37, 4.55, 1.25])    
    flux_table    = (1E-13) * np.array([15.3, 1.92, 4.3, 38, 34, 19.6, 3.36, 10.4, 6.27, 3.85, 5.25, 1.12, 1.26, 1.74, 8.69, 2.77, 5.96, 2.23])  
    gamma_table   = np.array([0.43, 0.72, 0.22, 0, 0.62, 0.38, 0.62, 0.96, 1.37, 0.56, 0.26, 0.98, 0.11, 0.28, -0.7, 1.16, 0.11, 0.96])  
    I67_table     = (1E-6) * np.array([2.51, 0.79, 3.5, 17, 5.92, 4.22, 2.24, 2.48, 3.04, 1.16, 1.52, 3.35, 2.66, 1.1, 6.84, 0.64, 1.93, 1.62])                 
    I69_table     = (1E-6) * np.array([1.47, 0.60, 2.69, 10.9, 6.57, 3.51, 1.15, 2.23, 2.48, 0.59, 0.93, 2.24, 0.75, 0.73, 5.34, 0.55, 1.89, 1.19])                 
    ratio_64_67   = np.array([0.68, 1.18, 0.13, 1.75, 1.26, 1.5, 1.21, 1.17, 1.93, 1.83, 0.86, 0.71, 1.29, 1.24, 1.98, 0.76, 0.84])              
   
    for i in range(num_cvs):
        #Chooses which index to use
        #idx = i % len(nh_table)   #Loops through the table if num_cvs > table size
        idx = np.random.randint(len(nh_table))
        
        #Extracts the corresponding values from the arrays
        nh = nh_table[idx]
        flux = flux_table[idx]
        gamma = gamma_table[idx]
        I_67 = I67_table[idx]
        I_69 = I69_table[idx]
        ratio = np.random.choice(ratio_64_67)  #Takes random value of the Xu data which has the 6.4/6.7 intensity ratio
        I_64 = ratio * I_67  #This computes the intensity of the 6.4 keV line (I64) from the ratio

        nh_table_vals.append(nh)
        flux_table_vals.append(flux)
        gamma_table_vals.append(gamma)
        I_67_table_vals.append(I_67)
        I_69_table_vals.append(I_69)
        I_64_table_vals.append(I_64)

    return nh_table_vals, flux_table_vals, gamma_table_vals, I_64_table_vals, I_67_table_vals, I_69_table_vals



In [226]:
def simulate_N_CVs(num_cvs, src_file_list, dest_path):

    """This function generates the simulated spectra for many CVs."""

    nh_table_vals, flux_table_vals, gamma_table_vals, I_64_table_vals, I_67_table_vals, I_69_table_vals = cv_paramvals_mondal(num_cvs)
    
    for i in range(num_cvs):  #Creates loop through the desired number of CVs to be generated
        
        sim_name = f"CV_sim_{i+1}.fak"   #Names the faked spectra simulations

        print(f"Simulating CV {i+1}/{num_cvs}")  #This lets us keep track of which CV spectra is being generated
        print(" NH:", nh_table_vals[i])
        print(" Flux:", flux_table_vals[i])
        print(" Gamma:", gamma_table_vals[i])
        print(" I_64/I_67/I_69:", I_64_table_vals[i], I_67_table_vals[i], I_69_table_vals[i])

        #Here, we call our previous function to generate a single CV spectra for a given i value
        sim_cv_from_jasmine(
            src_file        = src_file_list[i],
            sim_cv_name     = sim_name,
            nh_val          = nh_table_vals[i],
            gamma_val       = gamma_table_vals[i],
            flux_2_10       = flux_table_vals[i],
            norm1           = I_64_table_vals[i],
            norm2           = I_67_table_vals[i],
            norm3           = I_69_table_vals[i], 
            output_dir      = dest_path)

    print("\nAll CVs simulated.")
    return nh_table_vals, flux_table_vals, gamma_table_vals, I_64_table_vals, I_67_table_vals, I_69_table_vals



In [227]:
#num_msps, nh_vals, temp_vals, unabs_lx_vals, ew_64_vals, ew_67_vals, ew_70_vals, src_folder, sim_cv_folder='./', file_prefix='cv_'

In [228]:
def main_cv(num_cvs=10000, nh_abs_type='high', src_folder=None,
            sim_msp_folder='./'):
    """Main function to generate CV

    src_folder = xmm_obs_good2
    
    """
    # Change below line to Mondel function
    #nh_vals, temp_vals, lx_vals, ew_64_vals, ew_67_vals, ew_70_vals = ip_param_vals(10)
    
    if src_folder is None:
        src_folder = './Galactic_' + nh_abs_type + 'NH_combinedXMM/'  # Change
    dest_path = '/Volumes/RESEARCH/Source_Data/sim_cvs_test/'
    
    #cvs_sims_from_src(10, nh_vals, temp_vals, lx_vals, ew_64_vals,
    #                  ew_67_vals, ew_70_vals, src_folder,
    #                  dest_path, 'ip_')
    
    (common_files_pn, common_files_mos, only_pn_files, only_mos_files) = get_xmm_src_files(src_folder)
    pn_files = common_files_pn + only_pn_files
    mos_files = common_files_mos + only_mos_files
    
    params = simulate_N_CVs(10000, pn_files, dest_path) #once working properly change 10 to num_cvs in parameters for main_cv()
    nh_vals, flux_vals, gamma_vals, I64_vals, I67_vals, I69_vals = params
    cv_param_vals = np.column_stack(params)
    #cv_param_vals = np.column_stack(nh_vals, temp, lx_vals)
    np.savetxt(dest_path + 'paramfile.txt', cv_param_vals)
    #return nh_vals, temp_vals, lx_vals
    return params

In [229]:
np.random.randint(10)

9

In [230]:
get_xmm_src_files('/Volumes/Pavan_Work_SSD/GalacticBulge_Xrayclassify/data/Galactic_highNH_combinedXMM/')

([], [], [], [])

In [231]:
import os


In [232]:
os.chdir('/Volumes/RESEARCH/Source_Data/')
xspec.AllData.clear()
os.getcwd()  #run this before running main_cv

'/Volumes/RESEARCH/Source_Data'

In [233]:
main_cv()

Simulating CV 1/10
 NH: 0.24
 Flux: 1.7400000000000002e-13
 Gamma: 0.28
 I_64/I_67/I_69: 1.4300000000000002e-07 1.1e-06 7.3e-07
Computed gaussian norms (phot/cm^2/s):
 norm_6.4 = 1.4300000000000002e-07
 norm_6.7 = 1.1e-06
 norm_6.9 = 7.3e-07

1 spectrum  in use
 
Spectral Data File: ./Galactic_highNH_combinedXMM/200305401010017_PN_combined_src_grp1_403cts.ds  Spectrum 1
Net count rate (cts/s) for Spectrum:1  3.287e-03 +/- 2.697e-04 (62.3 % total)
 Assigned to Data Group 1 and Plot Group 1
  Noticed Channels:  1-339
  Telescope: XMM Instrument: EPIC  Channel Type: PI
  Exposure Time: 7.283e+04 sec
 Using fit statistic: chi
 Using Background File                ./Galactic_highNH_combinedXMM/200305401010017_PN_combined_bkg_grp.ds
  Background Exposure Time: 7.283e+04 sec
 Using Response (RMF) File            ./Galactic_highNH_combinedXMM/200305401010017_PN_combined_rsp_grp.ds for Source 1


Model TBabs<1>(pegpwrlw<2> + gaussian<3> + gaussian<4> + gaussian<5>) Source No.: 1   Active/On
Mod

([0.24, 1.46, 0.37, 0.76, 0.37, 6.22, 3.72, 1.07, 0.67, 0.24],
 [1.7400000000000002e-13,
  1.26e-13,
  2.7700000000000003e-13,
  8.689999999999999e-13,
  2.7700000000000003e-13,
  5.25e-13,
  4.3e-13,
  1.5300000000000001e-12,
  1.92e-13,
  1.7400000000000002e-13],
 [0.28, 0.11, 1.16, -0.7, 1.16, 0.26, 0.22, 0.43, 0.72, 0.28],
 [1.4300000000000002e-07,
  3.3516e-06,
  8.256000000000001e-07,
  8.002799999999999e-06,
  7.552e-07,
  1.7935999999999996e-06,
  2.94e-06,
  4.5933e-06,
  5.608999999999999e-07,
  2.178e-06],
 [1.1e-06,
  2.66e-06,
  6.4e-07,
  6.84e-06,
  6.4e-07,
  1.5199999999999998e-06,
  3.5e-06,
  2.5099999999999997e-06,
  7.9e-07,
  1.1e-06],
 [7.3e-07,
  7.5e-07,
  5.5e-07,
  5.34e-06,
  5.5e-07,
  9.3e-07,
  2.6899999999999997e-06,
  1.47e-06,
  6e-07,
  7.3e-07])

In [46]:
os.getcwd()

'/Volumes/RESEARCH'